# Feature Selection

Feature selection is the process of selecting the most important input features and removing irrelevant or redundant ones before training a machine learning model.

Instead of using all features, we keep only those that contribute the most to prediction.

## Why Feature Selection is Important?
- Improves model performance and accuracy
- Reduces overfitting
- Decreases training time
- Makes the model simpler and more interpretable

> Feature selection helps keep only useful features, making models faster, simpler, and more effective.

---

### 1. Filter Methods
Select features based on statistical measures (independent of the model).
- Correlation
- Chi-Square
- Mutual Information

### Import libraries

In [1]:
import pandas as pd
import seaborn as sns

### Load dataset

In [2]:
df = sns.load_dataset("titanic")

### Preview data

In [3]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


### Drop unnecessary columns

In [4]:
df = df.drop(columns=["alive", "who", "adult_male", "deck", "embark_town", "class"])

### Handle missing values

In [5]:
df = df.dropna()

### Encode categorical variables

In [6]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

### Define features and target

In [7]:
X = df.drop("survived", axis=1)
y = df["survived"]

### 1.1 Correlation Method

In [8]:
corr = df.corr()["survived"].sort_values(ascending=False)
print("Correlation with Target:\n", corr)

Correlation with Target:
 survived    1.000000
fare        0.266100
parch       0.095265
sibsp      -0.015523
age        -0.082446
embarked   -0.181979
alone      -0.199741
pclass     -0.356462
sex        -0.536762
Name: survived, dtype: float64


### 1.2 Chi-Square Test (Requires non-negative values)

In [9]:
from sklearn.feature_selection import chi2
chi_scores, p_values = chi2(X, y)
chi_df = pd.DataFrame({
    "Feature": X.columns,
    "Chi2 Score": chi_scores,
    "p-value": p_values
}).sort_values(by="Chi2 Score", ascending=False)
print("\nChi-Square Results:\n", chi_df)


Chi-Square Results:
     Feature   Chi2 Score       p-value
5      fare  4081.679420  0.000000e+00
1       sex    74.621277  5.702528e-18
2       age    34.246098  4.856483e-09
0    pclass    28.243213  1.069891e-07
7     alone    12.367930  4.367716e-04
4     parch    10.883500  9.702474e-04
6  embarked     8.956375  2.765030e-03
3     sibsp     0.288691  5.910606e-01


### 1.3. Mutual Information

In [10]:
from sklearn.feature_selection import mutual_info_classif
mi_scores = mutual_info_classif(X, y)
mi_df = pd.DataFrame({
    "Feature": X.columns,
    "MI Score": mi_scores
}).sort_values(by="MI Score", ascending=False)
print("\nMutual Information:\n", mi_df)


Mutual Information:
     Feature  MI Score
1       sex  0.132686
5      fare  0.130514
7     alone  0.049972
0    pclass  0.041886
4     parch  0.018817
2       age  0.000000
3     sibsp  0.000000
6  embarked  0.000000


---

### 2. Wrapper Methods
Select features by training models and evaluating performance.
- Recursive Feature Elimination (RFE)

### Import libraries

In [11]:
import pandas as pd
import seaborn as sns

### Load dataset

In [12]:
df = sns.load_dataset("titanic")

### Drop less useful columns

In [13]:
df = df.drop(columns=["alive", "who", "adult_male", "deck", "embark_town", "class"])

### Remove missing values

In [14]:
df = df.dropna()

### Encode categorical columns

In [15]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

### Define features and target

In [16]:
X = df.drop("survived", axis=1)
y = df["survived"]

### Make a model

In [17]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)

### 2.1 RFE (select top 5 features)

In [18]:
from sklearn.feature_selection import RFE
rfe = RFE(estimator=model, n_features_to_select=5)
rfe.fit(X, y)

,estimator,LogisticRegre...max_iter=1000)
,n_features_to_select,5
,step,1
,verbose,0
,importance_getter,'auto'
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1


### Results

In [19]:
selected_features = pd.DataFrame({
    "Feature": X.columns,
    "Selected": rfe.support_,
    "Rank": rfe.ranking_
}).sort_values(by="Rank")

print(selected_features)

    Feature  Selected  Rank
0    pclass      True     1
1       sex      True     1
3     sibsp      True     1
7     alone      True     1
6  embarked      True     1
4     parch     False     2
2       age     False     3
5      fare     False     4


---

### 3. Embedded Methods
Feature selection happens during model training.
- Lasso (L1 Regularization)
- Tree-based feature importance

### # Load dataset

In [20]:
import pandas as pd
df = pd.read_csv("House_price.csv")

### Drop Address (not useful for ML)

In [21]:
df = df.drop("Address", axis=1)

### Split features and target

In [22]:
X = df.drop("Price", axis=1)
y = df["Price"]

### 3.1 Low Variance Feature Removal

In [23]:
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(X)
print("Shape after Variance Threshold:", X_var.shape)

Shape after Variance Threshold: (4548, 5)


### 3.2 Lasso Regression 

In [24]:
from sklearn.linear_model import Lasso
lasso = Lasso(alpha=0.1)
lasso.fit(X, y)

lasso_importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lasso.coef_
}).sort_values(by="Coefficient", key=abs, ascending=False)

print("\nLasso Feature Importance:\n", lasso_importance)


Lasso Feature Importance:
               Feature    Coefficient
1           House Age  165341.628177
2     Number of Rooms  121144.827185
3  Number of Bedrooms    1458.445713
0    Avg. Area Income      21.609719
4     Area Population      15.188240


### 3.3 Random Forest Feature Importance

In [25]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state=42)
rf.fit(X, y)

rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)
print("\nRandom Forest Feature Importance:\n", rf_importance)


Random Forest Feature Importance:
               Feature  Importance
0    Avg. Area Income    0.432596
1           House Age    0.235583
4     Area Population    0.189313
2     Number of Rooms    0.125621
3  Number of Bedrooms    0.016887
